<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B08%5D%20-%20Ingenieria_de_Variables_II/%5B01%5D%20-%20Notebooks/E3_Visualiza_en_2D_con_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · Visualiza en 2D con PCA - Ingeniería de Variables II

## Introducción

No podemos dibujar un espacio de 60 dimensiones, pero sí podemos **proyectarlo a 2** con PCA
y echarle un vistazo. Es uno de los usos más útiles de la reducción de dimensionalidad:
**ver** los datos.

En este ejercicio proyectamos un dataset ancho a **2 componentes** y lo pintamos en un
scatter. La pregunta: **¿se ven grupos?**

## Objetivos del ejercicio

- Proyectar un dataset de muchas columnas a **2 componentes** con PCA.
- **Visualizar** los datos en 2D y observar si aparecen grupos.
- Entender que PCA **no usa el target**: la estructura sale sola de la varianza.
- Conectar con la idea de **clustering**.

## Descripción del dataset (sensores, dataset "ancho")

Para esta sesión usamos un dataset **sintético y reproducible** que imita un caso muy
común: **muchísimas columnas pero pocas dimensiones reales**. Piensa en cientos de
sensores que, en el fondo, miden unas pocas cosas (temperatura, presión, vibración...).

Lo generamos dentro del propio notebook con `generar_datos_anchos`, así que es
autocontenido en Colab. Por dentro:

- Hay unos pocos **factores latentes** (la "verdad" oculta) que generan la señal.
- Cada **columna `sensor_XXX`** es una mezcla de esos factores más algo de ruido, así que
  muchas columnas **dicen casi lo mismo** (están muy correlacionadas).
- Las columnas tienen **escalas muy distintas** a propósito (de 1 a 1000), para ver por
  qué hay que escalar antes de PCA.
- `target` es la etiqueta (la clase de cada fila).

> La idea de fondo de la clase: *más columnas no significa más información*. Aquí lo vemos
> en directo, porque la información real vive en muy pocas dimensiones.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

### 2. Un dataset ancho con 3 grupos

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_anchos(n=800, n_cols=60, n_latentes=5, n_clases=2,
                         separacion=2.5, ruido=0.6, semilla=42):
    # Genera un dataset "ancho": muchas columnas (sensores) pero POCAS dimensiones
    # reales. Unos pocos factores latentes generan casi toda la informacion; el
    # resto de columnas son mezclas de esos factores mas ruido. Asi PCA puede
    # recuperar la estructura con pocas componentes.
    rng = np.random.default_rng(semilla)

    # Centros de cada clase en el espacio latente (grupos separados)
    centros = rng.normal(scale=separacion, size=(n_clases, n_latentes))
    y = rng.integers(0, n_clases, size=n)
    Z = centros[y] + rng.normal(size=(n, n_latentes))      # factor latente de cada fila

    # Cada columna observada = mezcla ponderada de los factores latentes + ruido
    cargas = rng.normal(size=(n_latentes, n_cols))
    X = Z @ cargas + ruido * rng.normal(size=(n, n_cols))

    # Escalas MUY distintas por columna (para motivar StandardScaler antes de PCA)
    escalas = rng.uniform(1, 1000, size=n_cols)
    X = X * escalas

    cols = [f"sensor_{i:03d}" for i in range(n_cols)]
    df = pd.DataFrame(X, columns=cols)
    df["target"] = y
    return df

In [ ]:
df = generar_datos_anchos(n=900, n_cols=60, n_latentes=5, n_clases=3,
                          separacion=3.0, semilla=10)
X = df.drop(columns="target")
y = df["target"]
print("Dimensiones:", X.shape, "| Nº de grupos reales:", y.nunique())

### 3. Escalar y proyectar a 2 componentes

In [ ]:
X_esc = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=0)
X_2d = pca.fit_transform(X_esc)

var_2d = pca.explained_variance_ratio_.sum()
print("Con solo 2 componentes conservamos el", f"{var_2d*100:.1f}%", "de la varianza.")

### 4. Scatter en 2D

In [ ]:
plt.figure(figsize=(7, 6))
for clase in sorted(y.unique()):
    m = y == clase
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=15, alpha=0.7, label=f"grupo {clase}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"Proyección a 2D con PCA ({var_2d*100:.0f}% de varianza)")
plt.legend()
plt.tight_layout()
plt.show()

### 5. ¿Se ven grupos? (conexión con clustering)

Aunque coloreamos por `target` para comprobarlo, **PCA no ha usado las etiquetas**: ha
encontrado las direcciones de mayor varianza y los grupos han aparecido solos. Pintémoslo
otra vez **sin colores**, como lo vería un algoritmo no supervisado:

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], s=15, alpha=0.6, color="#34495e")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Mismos datos sin etiquetas: ¿cuántos grupos dirías?")
plt.tight_layout()
plt.show()

Justo esto es lo que hace el **clustering**: buscar grupos sin mirar el target. PCA es un
buen primer paso para **ver** la estructura antes de aplicar un k-means o un DBSCAN.

### Reflexión

1. ¿Cuánta varianza se conserva con solo 2 componentes? ¿Es suficiente para ver los grupos?
2. PCA no usa el target, ¿por qué aparecen igualmente los grupos?
3. ¿Qué pasaría si los grupos estuvieran menos separados (baja la `separacion`)?
4. ¿Cómo enlazarías esta visualización con un algoritmo de clustering?